# Jev as a Judge

A doctor's scheduling assistant turns messages into actions such as blocking time or
rescheduling appointments. Here, Jev checks whether the assistant understood the request.

This notebook runs Jev live on 21 labeled practice cases. To compare Jev with GPT-5 and
run confidence-based fallback, open [the comparison lab](jev_vs_llm_judge.ipynb).

## What are we judging?

The scheduling assistant reads a doctor's message and chooses an action. The judge sees
that message and the predicted action; it does not execute appointments or see the human answer.

**Failure category:** Failure to interpret indirect or underspecified requests.

**Binary question:** Did the router fail to interpret the indirect or underspecified scheduling request?

### The routing policy

| Intent | What the assistant can do |
| --- | --- |
| `LIST_APPOINTMENTS` | View existing appointments; not analyze historical patient counts. |
| `CANCEL_APPOINTMENTS` | Cancel all appointments in a date/time range. |
| `RESCHEDULE_APPOINTMENTS` | Move all appointments from one date/time range to another. |
| `BLOCK_SLOTS` | Make the doctor unavailable. |
| `UNBLOCK_SLOTS` | Restore availability. |
| `UPDATE_SPECIAL_SLOTS` | Add, extend or reduce working hours. |
| `CONVERSATIONAL_GREETING` | Respond to an opening social greeting. |
| `CLOSING_CONVERSATION` | Recognize a closing thanks, acknowledgment or goodbye. |
| `UNSUPPORTED_REQUEST` | A scheduling operation outside these capabilities, or a request whose operation remains unclear. |
| `IRRELEVANT_QUERY` | A request unrelated to the doctor, schedule, patients or hospital. |

Creating an appointment, or canceling/moving a specific appointment identified by patient
name or relative position, is unsupported. Missing a date, time or destination does **not**
make a recognizable supported action unsupported. When wording or missing calendar state
leaves multiple reasonable interpretations, accept a defensible prediction. Do not invent
prior blocks, working hours, or a rule for prioritizing multiple actions.

### The TRUE/FALSE rule

Return **TRUE** only when all three are established:

1. The message expresses the action indirectly, through shorthand or a misspelling, or with incomplete execution details. Ordinary politeness alone is insufficient.
2. The predicted intent is ruled out by every reasonable reading allowed by the policy.
3. The mismatch is a failure to recover that scheduling meaning.

Return **FALSE** when this failure is not established. FALSE includes correct routes,
defensible ambiguity, and different errors such as reversing explicit available/unavailable
wording or crossing an unsupported capability boundary. It does not mean “correct overall.”

Jev receives these rules and the same six worked examples below.

| Doctor's message | Router prediction | Judge | Why |
| --- | --- | --- | --- |
| I can work a half-day this Saturday. | `UNSUPPORTED_REQUEST` | **TRUE** | Working availability is recoverable despite missing hours. |
| I need to reschedule. | `RESCHEDULE_APPOINTMENTS` | **FALSE** | The rescheduling action was correctly recognized. |
| I can work late on the 3rd until 8 PM. | `UNSUPPORTED_REQUEST` | **TRUE** | The implied extension of working hours was missed. |
| I'm not working *this* Saturday. | `UNBLOCK_SLOTS` | **FALSE** | This is an explicit polarity error, a different category. |
| Move my afternoon. | `UNSUPPORTED_REQUEST` | **TRUE** | Recognizable rescheduling shorthand was rejected. |
| Move Mr. Johnson's appointment from Monday to Tuesday. | `UNSUPPORTED_REQUEST` | **FALSE** | Moving a named patient's appointment is unsupported. |

## Connect and load the cases

Supply a TypeSafe API key through environment variables, Colab Secrets, an ignored
`.env`, or hidden input. Each inference cell sends fresh requests.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab needs a checkout; locally, open this notebook inside the repository.
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    checkout = Path("/content/routing-evals-lab")
    if not checkout.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "codex/jev-routing-judge",
                        "https://github.com/The-Gen-Academy/Mastering-Agentic-AI-Week4-Session1.git",
                        str(checkout)], check=True)
    os.chdir(checkout)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "pandas", "python-dotenv"], check=True)

folders = [Path.cwd(), Path("routing_agent_evals"), Path("AI-Evals-Principles/routing_agent_evals")]
LAB = next(folder.resolve() for folder in folders if (folder / "judge_lab.py").exists())
os.chdir(LAB)
sys.path.insert(0, str(LAB))

In [ ]:
import pandas as pd
from IPython.display import display
from judge_lab import load_dataset, batch, comparison_metrics, always_false, resolved, save_json
from notebook_utils import setup_run, records_for_json
from prompts.llm_as_judge_prompt import FAILURE_CATEGORY, FAILURE_QUESTION

client, RUN = setup_run(LAB, stages=("jev",))
rows = load_dataset(LAB / "data", split="calibration")
print("Practice cases:", len(rows))
display(pd.DataFrame(rows)[["id", "user_query", "predicted_intent"]].head(3))

## Ask the binary question

Jev uses Choice options TRUE and FALSE. The response includes probabilities and native
confidence. The helper validates each answer; invalid output remains unresolved.

In [ ]:
jev = batch(rows, lambda row: client.call(row, "jev"))
display(pd.DataFrame([jev[rows[0]["id"]]]).reindex(columns=["label", "confidence", "probabilities", "error"]))

## Compare with human labels

Score the same resolved cases for Jev and the always-FALSE baseline. Inspect TRUE
precision and recall alongside overall agreement. Coverage and unresolved counts
show missing answers; unknown costs are left blank.

In [ ]:
comparison = pd.DataFrame(comparison_metrics(rows, {"Always FALSE": always_false(rows), "Jev": jev}))
table = comparison[["method", "scored_n", "coverage", "accuracy", "precision_TRUE", "recall_TRUE", "f1_TRUE", "FP", "FN", "unresolved", "cost_usd"]].copy()
table["cost_usd"] = table.cost_usd.where(comparison.unknown_cost_attempts.eq(0))
display(table.round(4))
comparison.to_csv(RUN / "comparison.csv", index=False)
pd.DataFrame(jev.values()).to_csv(RUN / "results.csv", index=False)
save_json(RUN / "summary.json", {"mode": "live", "category": FAILURE_CATEGORY,
    "question": FAILURE_QUESTION, "comparison": records_for_json(comparison),
    "evaluation_ids": [r["id"] for r in rows], "http_attempts": client.new_requests})
print("Outputs:", RUN)